# KRONOS2 vs KRONOS zero-shot cell typing: probes from saved embeddings (CPU)

Same pre-registered design as `Kronos2ZeroShot.ipynb` (see its first cell: arms, replicates 0-2, gate applied to each
encoder, head-to-head). `kronos2-zeroshot` v1 completed the embedding stage and then ran out of RAM in the pooled ('Both')
probes. This notebook runs the probe and gate stages only:
- KRONOS2 embeddings: dataset `kronos2-embeddings` (that run's saved outputs, with its calibration: normalisation variant
  and PCA). KRONOS (v1) embeddings: kernel `kronos-zeroshot`. Both are aligned to the cells by (image, object).
- Memory fix: pooled cohorts read feature rows from their parts instead of concatenating every table.
- CPU only; the logistic probes use the same LBFGS code on CPU.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, gc, traceback, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm as lgb
pip = [sys.executable, "-m", "pip", "install", "-q"]
if subprocess.run(pip + ["--only-binary=:all:", "imagecodecs==2024.9.22"]).returncode:
    subprocess.run(pip + ["--only-binary=:all:", "imagecodecs"], check=True)
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1" or os.path.exists(f"{IN}/SMOKE")
SEED = 20261013                                   # same seed and split rules as cell-type-graph / kronos-zeroshot
N_BOOT = 100 if SMOKE else 1000
MIN_CELLS = 200
COF = 5.0                                         # table features exactly as in cell-type-graph
N_FIT = 20_000 if SMOKE else 300_000
LGB = dict(n_estimators=30 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
REPS = [int(r) for r in os.environ.get("REPS", "0,1,2").split(",")]
SMOKE_IMAGES = 24
CROP, UP = 32, 2                                  # 32 px window at 1 µm/px, upsampled to a 64 px cell patch (as v1)
Q_NORM = 99.9
BATCH = 256
N_PCA, N_K = 256, 32
N_CAL_FIT, N_CAL_PCA = (8, 8) if SMOKE else (30, 40)
BUDGET_MIN = 660.0
dev = "cuda" if torch.cuda.is_available() else "cpu"
T0 = time.time()
FAILED = []

NUC = ["histone_h3", "dna1", "dna2"]
MARKERS = ["c_erb_b_2_her2", "cd20", "cd3", "cd45", "cd68", "cytokeratin_5", "cytokeratin_8_18",
           "estrogen_receptor_alpha", "ki_67", "pan_cytokeratin", "sma"]
FEATS = NUC + MARKERS
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
IMG_CHAN = {"Danenberg": D_CHAN, "Jackson": J_CHAN}
# KRONOS2 marker name -> source image channels (averaged); same 13 channels and order as v1
KCH = [("DAPI", ["dna1", "dna2"]), ("HISTONEH3", ["histone_h3"]), ("HER2", ["c_erb_b_2_her2"]), ("CD20", ["cd20"]),
       ("CD3", ["cd3"]), ("CD45", ["cd45"]), ("CD68", ["cd68"]), ("CK5", ["cytokeratin_5"]),
       ("CK8_18", ["cytokeratin_8_18"]), ("ERA", ["estrogen_receptor_alpha"]), ("KI67", ["ki_67"]),
       ("CYTOKERATIN", ["pan_cytokeratin"]), ("A-SMA", ["sma"])]
NOVEL = {"CK5": "Cytokeratin 5 (KRT5)", "CK8_18": "Cytokeratin 8/18 (KRT8/KRT18)"}   # not in the KRONOS2 vocabulary
NK = len(KCH)

COARSE = ["epithelial", "T cell", "B cell", "myeloid", "endothelial", "stroma"]
D_COARSE = {"CD8^{+} T cells": "T cell", "CD4^{+} T cells": "T cell", "T_{Reg} & T_{Ex}": "T cell",
            "B cells": "B cell",
            "Macrophages": "myeloid", "Granulocytes": "myeloid", "Macrophages & granulocytes": "myeloid",
            "Endothelial": "endothelial",
            "Fibroblasts": "stroma", "Fibroblasts FSP1^{+}": "stroma", "Myofibroblasts": "stroma",
            "Myofibroblasts PDPN^{+}": "stroma"}
J_COARSE = {1: "T cell", 2: "B cell", 4: "myeloid", 6: "myeloid", 7: "endothelial",
            **{m: "stroma" for m in range(8, 14)}, **{m: "epithelial" for m in range(14, 28)}}

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

print("SMOKE" if SMOKE else "FULL", "| device", dev, "| GPUs", torch.cuda.device_count(), "| torch", torch.__version__,
      "| reps", REPS)

## Cells, labels, splits and table features (as in `cell-type-graph`)

In [ ]:
P = {}
for p in g("cells.parquet"):
    names = pq.read_schema(p).names
    if "object_number" not in names or "published__h3k27me3" in names:
        continue
    key = "Danenberg" if "metabric_id" in names else "Jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p))
P["Danenberg"]["split"] = near("danenberg_patient_split.csv", P["Danenberg"]["cells"])
print(json.dumps(P, indent=1))

IMGP, MASKP = {"Danenberg": {}, "Jackson": {}}, {"Danenberg": {}, "Jackson": {}}
for p in g("*.tiff"):
    q = p.replace("\\", "/")
    if "keren" in q.lower():
        continue
    name = os.path.splitext(os.path.basename(p))[0]
    co = "Danenberg" if name.startswith("MB_") else "Jackson"
    (MASKP if "/masks/" in q else IMGP)[co][name] = p
print({c: (len(IMGP[c]), len(MASKP[c])) for c in IMGP})

def load(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "Danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    keep = n_per.index[n_per >= MIN_CELLS]
    keep = keep[keep.isin(list(set(IMGP[key]) & set(MASKP[key])))]
    if SMOKE:
        keep = np.sort(np.random.default_rng(SEED).choice(np.asarray(keep), SMOKE_IMAGES, replace=False))
    cells = cells[cells.image_name.isin(keep)].sort_values(["image_name", "object_number"]).reset_index(drop=True)
    n = len(cells)
    img, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img)) + 1, n]
    loc = pd.Series(np.arange(n), index=pd.MultiIndex.from_arrays([cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    CON = sp.coo_matrix((np.ones(int((ca != cb).sum())), (ca[ca != cb], cb[ca != cb])), shape=(n, n)).tocsr()
    CON = ((CON + CON.T) > 0).astype(np.float64).tocsr()
    cols = [D_TABLE.get(c, c) for c in FEATS] if key == "Danenberg" else FEATS
    X = np.c_[np.arcsinh(cells[cols].values.astype(np.float64) / COF),
              np.log(np.maximum(cells["area"].values, 1))].astype(np.float32)
    phen = cells.cell_phenotype.astype(str).values
    fine = sorted(set(phen))
    y_fine = pd.Series(phen).map({p: i for i, p in enumerate(fine)}).values.astype(np.int64)
    if key == "Danenberg":
        epi = cells.groupby(cells.cell_phenotype.astype(str)).is_epithelial.mean()
        cmap = {p: ("epithelial" if epi[p] > 0.5 else D_COARSE.get(p)) for p in fine}
    else:
        mc = cells.groupby(cells.cell_phenotype.astype(str)).metacluster.first()
        cmap = {p: J_COARSE.get(int(mc[p])) for p in fine}
    coarse_of = np.array([COARSE.index(cmap[p]) if cmap[p] else -1 for p in fine], np.int64)
    return dict(key=key, n=n, starts=starts, n_img=len(img_names), img_names=np.asarray(img_names),
                obj=cells.object_number.values.astype(np.int64), patient=cells[pid].astype(str).values,
                xy=cells[["x", "y"]].values.astype(np.float64), CON=CON, X=X, fine=fine,
                y={"fine": y_fine, "coarse": coarse_of[y_fine]})

CO = {k: load(k) for k in ("Jackson", "Danenberg")}
CJ, CD = CO["Jackson"], CO["Danenberg"]
split0 = pd.read_csv(P["Danenberg"]["split"]).set_index("metabric_id").split
CD["split0"] = pd.Series(CD["patient"]).map(split0).fillna("none").values

def harm(X):
    med = np.nanmedian(X, axis=0)
    q05, q95 = np.nanpercentile(X, [5, 95], axis=0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X, 0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

def knn10(C, k=10):
    rows, cols = [], []
    for a, b in zip(C["starts"][:-1], C["starts"][1:]):
        kk = min(k, b - a - 1)
        _, nb = cKDTree(C["xy"][a:b]).query(C["xy"][a:b], k=kk + 1)
        rows.append(np.repeat(np.arange(a, b), kk)); cols.append(nb[:, 1:].ravel() + a)
    A = sp.coo_matrix((np.ones(sum(map(len, rows)), np.float32), (np.concatenate(rows), np.concatenate(cols))),
                      shape=(C["n"], C["n"])).tocsr()
    A = ((A + A.T) > 0).astype(np.float32).tocsr()
    A.sort_indices()
    return A

def r30_count(C):
    out = np.zeros(C["n"], np.int64)
    for a, b in zip(C["starts"][:-1], C["starts"][1:]):
        out[a:b] = cKDTree(C["xy"][a:b]).query_ball_point(C["xy"][a:b], 30.0, return_length=True) - 1
    return out

def nbr_features(C, A):
    X = C["XS"]["cohort"]
    deg = np.diff(A.indptr).astype(np.float32)[:, None]
    mean = (A @ X) / deg
    std = np.sqrt(np.maximum((A @ (X * X)) / deg - mean * mean, 0))
    mx = np.maximum.reduceat(X[A.indices], A.indptr[:-1], axis=0)
    hop2 = (A @ mean) / deg
    dst = np.repeat(np.arange(C["n"]), np.diff(A.indptr))
    dist = np.bincount(dst, np.linalg.norm(C["xy"][A.indices] - C["xy"][dst], axis=1), C["n"]) / deg.ravel()
    return np.c_[X, mean, mx, std, hop2, np.log1p(r30_count(C)), np.log1p(np.diff(C["CON"].indptr)), dist / 30.0].astype(np.float32)

for key, C in CO.items():
    C["XS"] = {"cohort": np.nan_to_num(harm(C["X"]))}
    C["XS"]["nbr"] = nbr_features(C, knn10(C))
    print(f"{key}: {C['n']:,} cells, {C['n_img']} images, {len(set(C['patient']))} patients, {len(C['fine'])} fine labels")
tick("tables, labels and neighbour features")

## Splits (replicate 0 decides the calibration)

In [ ]:
D_TRVA_PAT = np.unique(CD["patient"][CD["split0"] != "test"])

def split_images(C, cell_split):
    s = cell_split[C["starts"][:-1]]
    return {k: np.flatnonzero(s == k) for k in ("train", "val", "test")}

def make_splits(REP):
    jp, jcode = np.unique(CJ["patient"], return_inverse=True)
    u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
    J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
    uv = np.random.default_rng(SEED + 21 + REP).random(len(D_TRVA_PAT))
    dval = set(D_TRVA_PAT[uv < 0.125])
    D_SPLIT = np.where(CD["split0"] == "test", "test", np.where(pd.Series(CD["patient"]).isin(dval).values, "val", "train"))
    IMG = {"Jackson": split_images(CJ, J_SPLIT), "Danenberg": split_images(CD, D_SPLIT)}
    IMG["Both"] = {s: np.r_[IMG["Jackson"][s], IMG["Danenberg"][s] + CJ["n_img"]] for s in ("train", "val")}
    return IMG

def cells_of(C, imgs):
    imgs = np.asarray(imgs, np.int64)
    if not len(imgs):
        return np.zeros(0, np.int64)
    return np.concatenate([np.arange(C["starts"][i], C["starts"][i + 1]) for i in imgs])

IMG0 = make_splits(0)
for c in ("Jackson", "Danenberg"):
    print(c, {s: len(v) for s, v in IMG0[c].items()}, "images")

## Saved embeddings

In [ ]:
# Saved embeddings: KRONOS2 (dataset kronos2-embeddings) and KRONOS v1 (kernel kronos-zeroshot), aligned by (image, object)
D_EMB, D_V1 = 768, 384
CAL = json.load(open(g("kronos2_calibration.json")[0]))
VARIANT = CAL["variant"]
K2_INFO = CAL.get("kronos2", {})

def aligned(emb_name, cells_name, C, key):
    cells1 = pd.read_parquet(g(cells_name)[0])
    pos = pd.Series(np.arange(len(cells1)), index=pd.MultiIndex.from_arrays([cells1.image_name.values, cells1.object_number.values]))
    names = C["img_names"][np.repeat(np.arange(C["n_img"]), np.diff(C["starts"]))]
    ix = pos.reindex(pd.MultiIndex.from_arrays([names, C["obj"]])).values
    assert not np.isnan(ix).any(), f"{key}: cells missing from {emb_name}"
    ix = ix.astype(np.int64)
    if not SMOKE:
        assert (ix == np.arange(C["n"])).all(), f"{key}: cell order differs in {emb_name}"
    return np.load(g(emb_name)[0], mmap_mode="r")[ix]

EMB, V1 = {}, {}
for key, C in CO.items():
    EMB[key] = aligned(f"kronos2_emb_{key.lower()}.npy", f"kronos2_cells_{key.lower()}.parquet", C, key)
    V1[key] = aligned(f"kronos_emb_{key.lower()}.npy", f"kronos_cells_{key.lower()}.parquet", C, key)
    assert EMB[key].shape[1] == D_EMB + N_PCA and V1[key].shape[1] == D_V1 + N_PCA
    print(f"{key}: KRONOS2 {EMB[key].shape} finite {np.isfinite(EMB[key].astype(np.float32)).all(1).mean():.4f} | "
          f"KRONOS {V1[key].shape} finite {np.isfinite(V1[key].astype(np.float32)).all(1).mean():.4f}")
print("calibration:", json.dumps({k: v for k, v in CAL.items() if k != "kronos2"}))
tick("embeddings loaded")

## Probe helpers (from the calibration cell)

In [ ]:
def scale_fit(Z):
    mu, sd = np.nanmean(Z, 0), np.nanstd(Z, 0) + 1e-6
    return lambda A: np.nan_to_num((A - mu) / sd).astype(np.float32)

def fit_logistic(Z, y, ncls, wds, Zv, yv, seed):
    """Class-weighted multinomial logistic regression (GPU, full batch, LBFGS); weight decay chosen on (Zv, yv)."""
    torch.manual_seed(seed)
    Zt, yt = torch.tensor(Z, device=dev), torch.tensor(y, device=dev)
    w = torch.tensor(np.bincount(y, minlength=ncls), device=dev, dtype=torch.float32)
    w = torch.where(w > 0, w.sum() / (ncls * w.clamp(min=1)), torch.zeros_like(w))
    best = None
    for wd in wds:
        lin = nn.Linear(Z.shape[1], ncls).to(dev)
        opt = torch.optim.LBFGS(lin.parameters(), lr=1, max_iter=200 if not SMOKE else 50, history_size=20,
                                line_search_fn="strong_wolfe")
        def closure():
            opt.zero_grad()
            loss = F.cross_entropy(lin(Zt), yt, weight=w) + wd * (lin.weight ** 2).sum()
            loss.backward()
            return loss
        opt.step(closure)
        with torch.no_grad():
            pv = lin(torch.tensor(Zv, device=dev)).argmax(1).cpu().numpy()
        f1 = macro_f1(yv, pv, ncls)
        if best is None or f1 > best[0]:
            best = (f1, wd, lin)
    return best

def macro_f1(y, p, ncls):
    cm = np.bincount(y * ncls + p, minlength=ncls * ncls).reshape(ncls, ncls).astype(float)
    tp, sup, prd = np.diag(cm), cm.sum(1), cm.sum(0)
    f1 = np.where(sup + prd > 0, 2 * tp / np.maximum(sup + prd, 1), 0)
    return float(f1[sup > 0].mean())

## Probes: cell typing, KRONOS2 and KRONOS vs the cell-table models (replicates 0-2)

In [ ]:
def k32_of(Z, d_cls):
    k = np.nan_to_num(Z[:, d_cls:d_cls + N_K].astype(np.float32))
    return (k - np.median(k, axis=0)) / (np.percentile(k, 95, axis=0) - np.percentile(k, 5, axis=0) + 1e-6)

ENC = {"KRONOS linear": "v1", "KRONOS2 linear": "v2"}
for key, C in CO.items():
    C["KF"] = {"v1": V1[key], "v2": EMB[key]}                                     # float16; rows are gathered when used
    k1, k2_ = k32_of(V1[key], D_V1), k32_of(EMB[key], D_EMB)
    C["XS"]["k32"] = np.c_[C["XS"]["cohort"], k1].astype(np.float32)
    C["XS"]["nbrk32"] = np.c_[C["XS"]["nbr"], k1].astype(np.float32)
    C["XS"]["k2_32"] = np.c_[C["XS"]["cohort"], k2_].astype(np.float32)
    C["XS"]["nbrk2_32"] = np.c_[C["XS"]["nbr"], k2_].astype(np.float32)
TABLE_ARMS = {"Table": "cohort", "Table + nbr": "nbr", "Table + K32": "k32", "Table + nbr + K32": "nbrk32",
              "Table + K2-32": "k2_32", "Table + nbr + K2-32": "nbrk2_32"}
FINE_ARMS = ("Table + nbr", "Table + nbr + K32", "Table + nbr + K2-32")
PAIRS = [(m, b) for m in list(TABLE_ARMS) + list(ENC) for b in ("Table", "Table + nbr") if m != b] + \
        [("KRONOS2 linear", "KRONOS linear"), ("Table + K2-32", "Table + K32"), ("Table + nbr + K2-32", "Table + nbr + K32")]

def combine(A, B):
    return dict(key="Both", n=A["n"] + B["n"], n_img=A["n_img"] + B["n_img"], starts=np.r_[A["starts"][:-1], B["starts"] + A["n"]],
                patient=np.r_[A["patient"], B["patient"]], cohort=np.r_[np.zeros(A["n"], np.int8), np.ones(B["n"], np.int8)],
                parts=(A, B),                               # feature rows are read from the parts (xs_rows, kf_rows)
                y={"coarse": np.r_[A["y"]["coarse"], B["y"]["coarse"]]})

def xs_rows(C, xkey, rows):
    """Feature rows; pooled cohorts are read from their parts instead of concatenating every table (memory)."""
    if C["key"] != "Both":
        return C["XS"][xkey][rows]
    A, B = C["parts"]
    out = np.empty((len(rows), A["XS"][xkey].shape[1]), np.float32)
    ia = rows < A["n"]
    out[ia] = A["XS"][xkey][rows[ia]]
    out[~ia] = B["XS"][xkey][rows[~ia] - A["n"]]
    return out

def kf_rows(C, tag, rows):
    """Encoder embedding of the given rows as float32 (NaN -> 0, as in v1); pooled cohorts are read from their parts."""
    if C["key"] != "Both":
        return np.nan_to_num(C["KF"][tag][rows].astype(np.float32))
    A, B = C["parts"]
    out = np.empty((len(rows), A["KF"][tag].shape[1]), np.float32)
    ia = rows < A["n"]
    out[ia] = A["KF"][tag][rows[ia]]
    out[~ia] = B["KF"][tag][rows[~ia] - A["n"]]
    return np.nan_to_num(out)

def class_weights(y, ncls):
    c = np.bincount(y, minlength=ncls).astype(float)
    return np.where(c > 0, c.sum() / (ncls * np.maximum(c, 1)), 0.0)

def train_rows(C, task, imgs, seed):
    idx = cells_of(C, imgs)
    idx = idx[C["y"][task][idx] >= 0]
    rs = np.random.default_rng(seed)
    pools = [idx[C["cohort"][idx] == c] for c in (0, 1)] if C["key"] == "Both" else [idx]
    return np.sort(np.concatenate([rs.choice(p, min(N_FIT, len(p)), replace=False) for p in pools if len(p)]))

def fit_table(xkey, C, task, ncls, tr, seed):
    fit = train_rows(C, task, tr, seed)
    y = C["y"][task][fit]
    return lgb.LGBMClassifier(random_state=seed, **LGB).fit(xs_rows(C, xkey, fit), y, sample_weight=class_weights(y, ncls)[y])

def predict_rows(C, imgs, task):
    node = cells_of(C, imgs)
    return node[C["y"][task][node] >= 0]

def scorer(C, imgs, task, ncls, seed):
    node = predict_rows(C, imgs, task)
    pats, pc = np.unique(C["patient"][node], return_inverse=True)
    W = np.random.default_rng(seed).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
    y = C["y"][task][node]
    return dict(node=node, y=y, pc=pc, npat=len(pats), W=W, ncls=ncls, present=np.bincount(y, minlength=ncls) > 0)

def metrics(Mx, ncls, present):
    Mx = Mx.reshape(-1, ncls, ncls)
    tp = np.einsum("bii->bi", Mx)
    sup, prd = Mx.sum(2), Mx.sum(1)
    with np.errstate(invalid="ignore", divide="ignore"):
        f1 = np.where(sup + prd > 0, 2 * tp / (sup + prd), 0.0)
        rec = np.where(sup > 0, tp / sup, np.nan)
        out = dict(macro_f1=f1[:, present].mean(1), bal_acc=np.nanmean(rec[:, present], 1), acc=tp.sum(1) / Mx.sum((1, 2)))
    return out, f1

def score(S, pred_full):
    p = pred_full[S["node"]]
    assert (p >= 0).all(), "missing predictions"
    k = S["ncls"]
    cm = np.bincount((S["pc"] * k + S["y"]) * k + p, minlength=S["npat"] * k * k).reshape(S["npat"], k * k).astype(np.float64)
    pt, f1pt = metrics(cm.sum(0, keepdims=True), k, S["present"])
    bt, _ = metrics(S["W"] @ cm, k, S["present"])
    return {m: float(v[0]) for m, v in pt.items()}, bt, f1pt[0]

def ci(v):
    v = np.asarray(v)[np.isfinite(v)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

UNITS = [("coarse", "Jackson"), ("coarse", "Danenberg"), ("coarse", "Both"), ("fine", "Jackson"), ("fine", "Danenberg")]
ROWS, DIFF, PCL = [], [], []
for REP in REPS:
    IMG = IMG0 if REP == 0 else make_splits(REP)
    SCR = {(task, t): scorer(CO[t], IMG[t]["test"], task, len(COARSE) if task == "coarse" else len(CO[t]["fine"]),
                             SEED + 31 + REP + 1000 * (t == "Danenberg"))
           for task in ("coarse", "fine") for t in ("Jackson", "Danenberg")}
    BOOT = {}
    for task, src in UNITS:
        if (time.time() - T0) / 60 > BUDGET_MIN:
            FAILED.append(f"budget: rep {REP} {task} {src}")
            continue
        C = combine(CJ, CD) if src == "Both" else CO[src]
        ncls = len(COARSE) if task == "coarse" else len(CO[src]["fine"])
        tests = ["Jackson", "Danenberg"] if task == "coarse" else [src]
        try:
            preds = {}
            arms = TABLE_ARMS if task == "coarse" else {k_: TABLE_ARMS[k_] for k_ in FINE_ARMS}
            for name, xkey in arms.items():
                mdl = fit_table(xkey, C, task, ncls, IMG[src]["train"], SEED + REP)
                for t in tests:
                    pr = np.full(CO[t]["n"], -1, np.int64)
                    node = predict_rows(CO[t], IMG[t]["test"], task)
                    pr[node] = mdl.classes_[mdl.predict_proba(CO[t]["XS"][xkey][node]).argmax(1)]
                    preds[(name, t)] = pr
            fit = train_rows(C, task, IMG[src]["train"], SEED + REP)
            vrows = np.concatenate([predict_rows(CO[s_], IMG[s_]["val"], task) + (CJ["n"] if src == "Both" and s_ == "Danenberg" else 0)
                                    for s_ in (["Jackson", "Danenberg"] if src == "Both" else [src])])
            for name, tag in ENC.items():
                Zf = kf_rows(C, tag, fit)
                sc = scale_fit(Zf)
                f1v, wd, lin = fit_logistic(sc(Zf), C["y"][task][fit], ncls, [1e-4, 1e-3, 1e-2],
                                            sc(kf_rows(C, tag, vrows)), C["y"][task][vrows], SEED + REP)
                del Zf
                for t in tests:
                    pr = np.full(CO[t]["n"], -1, np.int64)
                    node = predict_rows(CO[t], IMG[t]["test"], task)
                    with torch.no_grad():
                        pr[node] = lin(torch.tensor(sc(kf_rows(CO[t], tag, node)), device=dev)).argmax(1).cpu().numpy()
                    preds[(name, t)] = pr
                del lin
            for (name, t), pr in preds.items():
                pt, bt, f1c = score(SCR[(task, t)], pr)
                BOOT[(task, src, t, name)] = bt
                for m_, v in pt.items():
                    lo, hi = ci(bt[m_])
                    ROWS.append(dict(rep=REP, task=task, train=src, test=t, model=name, metric=m_, value=v, lo=lo, hi=hi))
                labs = COARSE if task == "coarse" else CO[t]["fine"]
                for c_ in np.flatnonzero(SCR[(task, t)]["present"]):
                    PCL.append(dict(rep=REP, task=task, train=src, test=t, model=name, label=labs[c_], f1=float(f1c[c_])))
            tick(f"rep {REP} {task} / train {src}: " + ", ".join(
                f"{n_}@{t[0]} {score(SCR[(task, t)], p_)[0]['macro_f1']:.3f}" for (n_, t), p_ in preds.items()))
            if src == "Both":
                del C
                gc.collect()
        except Exception:
            print(f"!!! rep {REP} {task} {src} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(f"rep {REP} {task} {src}")
    R_ = pd.DataFrame(ROWS)
    R_ = R_[(R_.rep == REP) & (R_.metric == "macro_f1")]
    for (task, src, t, name), bt in BOOT.items():
        for m_, base in PAIRS:
            kb = (task, src, t, base)
            if m_ != name or kb not in BOOT:
                continue
            d = R_[(R_.task == task) & (R_.train == src) & (R_.test == t)].set_index("model").value
            lo, hi = ci(bt["macro_f1"] - BOOT[kb]["macro_f1"])
            DIFF.append(dict(rep=REP, task=task, train=src, test=t, model=name, base=base, diff=float(d[name] - d[base]),
                             lo=lo, hi=hi, call="better" if lo > 0 else "worse" if hi < 0 else "n.s."))
RES, DIFFD = pd.DataFrame(ROWS), pd.DataFrame(DIFF)
RES.to_csv(f"{OUT}/scores.csv", index=False)
DIFFD.to_csv(f"{OUT}/diffs.csv", index=False)
pd.DataFrame(PCL).to_csv(f"{OUT}/perclass.csv", index=False)
with pd.option_context("display.width", 300, "display.max_columns", 20):
    f1t = RES[RES.metric == "macro_f1"]
    print(f1t.pivot_table(index=["rep", "task", "train", "test"], columns="model", values="value").round(3).to_string())
    print("\nmean (SD) over replicates")
    agg = f1t.groupby(["task", "train", "test", "model"]).value.agg(["mean", "std"]).round(3)
    print(agg.unstack("model").to_string())
    print(DIFFD[DIFFD.base.isin(["Table + nbr", "KRONOS linear", "Table + nbr + K32", "Table + K32"])].round(3).to_string(index=False))

## Gate for adaptation (both encoders) and the head-to-head

In [ ]:
GATE_ARMS = {"KRONOS": ("KRONOS linear", "Table + nbr + K32"), "KRONOS2": ("KRONOS2 linear", "Table + nbr + K2-32")}
f1 = RES[RES.metric == "macro_f1"]
reps_run = sorted(f1.rep.unique().tolist())
gate = {}
for enc, (lin_arm, k_arm) in GATE_ARMS.items():
    per_rep = {}
    for REP in reps_run:
        a = b = False
        det = []
        fr = f1[f1.rep == REP]
        for src in ("Jackson", "Danenberg"):
            s = fr[(fr.task == "coarse") & (fr.train == src) & (fr.test == src)].set_index("model").value
            if lin_arm in s and "Table + nbr" in s:
                det.append(f"(a) {src[0]}->{src[0]}: {lin_arm} {s[lin_arm]:.3f} vs Table + nbr {s['Table + nbr']:.3f} "
                           f"({s[lin_arm] - s['Table + nbr']:+.3f})")
                a |= bool(s[lin_arm] >= s["Table + nbr"] - 0.05)
        bb = DIFFD[(DIFFD.rep == REP) & (DIFFD.task == "coarse") & (DIFFD.model == k_arm) & (DIFFD.base == "Table + nbr")]
        for r in bb.itertuples():
            det.append(f"(b) {r.train[0]}->{r.test[0]}: {k_arm} {r.diff:+.3f} [{r.lo:+.3f}, {r.hi:+.3f}]")
            b |= bool(r.call == "better" and r.diff >= 0.01)
        per_rep[int(REP)] = dict(a=a, b=b, passed=a or b, details=det)
    gate[enc] = dict(per_replicate=per_rep,
                     rep0_rule_v1=per_rep.get(0, {}).get("passed"),
                     passes_every_replicate=bool(len(per_rep) == len(REPS) and all(v["passed"] for v in per_rep.values())),
                     replicates_run=reps_run)
    gate[enc]["proceed_to_adaptation"] = gate[enc]["passes_every_replicate"]

h2h = DIFFD[(DIFFD.task == "coarse") & (((DIFFD.model == "KRONOS2 linear") & (DIFFD.base == "KRONOS linear")) |
                                       ((DIFFD.model == "Table + nbr + K2-32") & (DIFFD.base == "Table + nbr + K32")))]
H2H = h2h.groupby(["model", "train", "test"]).agg(mean_diff=("diff", "mean"), min_diff=("diff", "min"), max_diff=("diff", "max"),
                                                  n_better=("call", lambda c: int((c == "better").sum())),
                                                  n_worse=("call", lambda c: int((c == "worse").sum())), n=("call", "size")).round(4)
H2H.to_csv(f"{OUT}/head_to_head.csv")
print(H2H.to_string())
json.dump(dict(gate=gate, calibration=CAL, variant=VARIANT, reps=REPS, failed=FAILED, smoke=SMOKE,
               minutes=(time.time() - T0) / 60, kronos2=K2_INFO), open(f"{OUT}/summary.json", "w"), indent=1)
print(json.dumps(gate, indent=1))
print("failed blocks:", FAILED if FAILED else "none")
tick("done")